# 05 — Token benchmark

Compares context sizes, memory length and providers on the same fixture candles.

**What is measured** per configuration: input, output, cached and reasoning tokens
(provider-reported where available; `None` = not reported, never 0), the estimated
tokens attributable to memory, latency, estimated cost and structured-output validity.

In MOCK mode the token counts are the prompt's real size (chars/4, deterministic), so
the *relative* effect of each knob is measured exactly; latency and validity are not
meaningful. REAL_LLM runs are exploratory observations, saved for comparison.

Configure REAL_LLM providers with `QBS_AGENT_BENCH_PROVIDERS`, e.g.
`gemini:gemini-2.5-flash,anthropic:claude-sonnet-5-5` (each needs its key/package).

**Modes.** `MOCK` (default) makes zero external LLM calls. `REAL_LLM` needs
`QBS_AGENT_TEST_MODE=REAL_LLM` **and** `QBS_AGENT_TEST_ALLOW_PAID=1`, prints a cost
estimate first, and stops at `QBS_AGENT_TEST_MAX_CALLS` (default 20).

**Isolation.** Every run uses a fresh temporary database (`tk.workspace()`); the
production config (`var/agent/config.json`) and log (`var/agent/agent.db`) are never
read or written, and nothing here can reach `qbs.live` or a broker.

**Dependencies.** `pip install -r requirements.txt jupyter` (MOCK). REAL_LLM also needs
the selected provider's package and key (see docs/TRADING_AGENT.md).

In [1]:
import os, sys, pathlib, json, logging
here = pathlib.Path.cwd()
ROOT = next(p for p in [here, *here.parents] if (p / "qbs" / "trading_agent").is_dir())
sys.path.insert(0, str(ROOT))
logging.basicConfig(level=logging.WARNING)

import pandas as pd
from qbs.trading_agent import testkit as tk

MODE = tk.test_mode()          # MOCK unless QBS_AGENT_TEST_MODE=REAL_LLM and QBS_AGENT_TEST_ALLOW_PAID=1
print("repo:", ROOT)
print("MODE:", MODE, "| paid-call cap:", tk.max_calls() if MODE == tk.REAL_LLM else 0)

repo: /home/user/qqq_boxx_strategies
MODE: MOCK | paid-call cap: 0


In [2]:
import itertools, time
from qbs.trading_agent import session as ss, store
from qbs.trading_agent.config import MarketContextConfig
from qbs.trading_agent.engine import TradingAgentEngine
from qbs.trading_agent.market_context import MarketContextBuilder

fx = tk.load_fixture(); day = fx.replay_date
# Seven candles, so a 5-decision history window actually fills up.
CANDLES = [tk.et(day, h, m) for h, m in ((10, 0), (10, 15), (10, 30), (10, 45), (11, 0), (11, 15), (11, 30))]
GRID = {"raw_15m": [4, 8, 16], "candles_15m": [16, 32], "decision_history": [3, 5]}
if MODE == tk.MOCK:
    PROVIDERS = [("mock", "mock-model")]
else:
    spec = os.environ.get("QBS_AGENT_BENCH_PROVIDERS") or f'{os.environ.get("QBS_AGENT_PROVIDER", "gemini")}:{os.environ.get("QBS_AGENT_MODEL", "")}'
    PROVIDERS = [tuple(p.split(":", 1)) for p in spec.split(",")]
    GRID = {"raw_15m": [4, 8], "candles_15m": [32], "decision_history": [3]}   # keep paid runs small
    CANDLES = CANDLES[:2]
combos = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
n_calls = len(combos) * len(PROVIDERS) * len(CANDLES) * len(fx.symbols)
print(len(combos), "configs x", len(PROVIDERS), "providers ->", n_calls, "LLM calls")
if MODE == tk.REAL_LLM:
    for prov, model in PROVIDERS:
        w = tk.workspace(MODE, provider=prov, model=model)
        print("ABOUT TO SPEND:", tk.cost_preview(w.cfg, n_calls // len(PROVIDERS)))
    assert n_calls <= tk.max_calls() * len(PROVIDERS), "raise QBS_AGENT_TEST_MAX_CALLS or shrink GRID"

12 configs x 1 providers -> 252 LLM calls


In [3]:
def num(col):
    """NULL (not reported) stays NaN; a sum over nothing stays NaN, not 0."""
    return pd.to_numeric(col, errors="coerce")

def run(combo, provider, model):
    mc = MarketContextConfig(raw_15m=combo["raw_15m"], candles_15m=combo["candles_15m"])
    ws = tk.workspace(MODE, provider=provider, model=model if provider != "mock" else "",
                      market_context=mc, decision_history=combo["decision_history"])
    clock = tk.Clock(tk.et(day, 9, 0))
    ss.save_daily_prompt(ws.cfg, fx.prompt, clock()); ss.authorize(ws.cfg, clock())
    clock.set(tk.et(day, 9, 40))
    sess, _ = ss.activate(ws.cfg, clock()); sess.transition(ss.RUNNING, "bench")
    eng = TradingAgentEngine(ws.cfg, sess, tk.make_provider(MODE, ws.cfg),
                             MarketContextBuilder(tk.FixtureBars(fx, clock), mc))
    for c in CANDLES:
        clock.set(c + pd.Timedelta(seconds=30)); eng.run_cycle(c)
    r = pd.DataFrame(store.requests_for(ws.cfg.db_path))
    return {**combo, "provider": provider, "model": eng.model, "requests": len(r),
            "input_tokens_mean": num(r.input_tokens).mean(), "output_tokens_mean": num(r.output_tokens).mean(),
            "cached_input_tokens_sum": num(r.cached_input_tokens).sum(min_count=1),
            "reasoning_tokens_sum": num(r.reasoning_tokens).sum(min_count=1),
            "est_input_tokens_mean": r.estimated_input_tokens.mean(),
            "memory_tokens_mean": r.memory_tokens_estimated.mean(),
            "latency_ms_mean": r.latency_ms.mean(), "valid_rate": (r.validation_status == "VALID").mean(),
            "cost_usd_sum": num(r.total_cost).sum(min_count=1), "usage_source": ",".join(sorted(set(r.usage_source)))}

results = pd.DataFrame([run(c, p, m) for p, m in PROVIDERS for c in combos])
results

,raw_15m,candles_15m,decision_history,provider,model,requests,input_tokens_mean,output_tokens_mean,cached_input_tokens_sum,reasoning_tokens_sum,est_input_tokens_mean,memory_tokens_mean,latency_ms_mean,valid_rate,cost_usd_sum,usage_source
0,4,16,3,mock,mock-model,21,1884.000000,98.380952,NaN,NaN,1884.000000,257.238095,0.676190,1.0,NaN,provider
1,4,16,5,mock,mock-model,21,1912.904762,98.380952,NaN,NaN,1912.904762,286.000000,1.585714,1.0,NaN,provider
2,4,32,3,mock,mock-model,21,1896.904762,98.380952,NaN,NaN,1896.904762,262.904762,0.633333,1.0,NaN,provider
3,4,32,5,mock,mock-model,21,1925.714286,98.380952,NaN,NaN,1925.714286,291.761905,0.633333,1.0,NaN,provider
4,8,16,3,mock,mock-model,21,1933.571429,98.380952,NaN,NaN,1933.571429,257.238095,0.685714,1.0,NaN,provider
5,8,16,5,mock,mock-model,21,1962.333333,98.380952,NaN,NaN,1962.333333,286.000000,0.738095,1.0,NaN,provider
6,8,32,3,mock,mock-model,21,1946.380952,98.380952,NaN,NaN,1946.380952,262.904762,0.909524,1.0,NaN,provider
7,8,32,5,mock,mock-model,21,1975.238095,98.380952,NaN,NaN,1975.238095,291.761905,0.671429,1.0,NaN,provider
8,16,16,3,mock,mock-model,21,2032.904762,98.380952,NaN,NaN,2032.904762,257.238095,0.776190,1.0,NaN,provider
9,16,16,5,mock,mock-model,21,2061.761905,98.380952,NaN,NaN,2061.761905,286.000000,0.714286,1.0,NaN,provider


## Effect of each knob on input tokens

In [4]:
for knob in GRID:
    if len(GRID[knob]) > 1:
        display(results.groupby(knob)[["est_input_tokens_mean", "memory_tokens_mean"]].mean().round(0))
check = tk.Check("05 token benchmark")
r8 = results[results.raw_15m == 8].est_input_tokens_mean.mean()
r4 = results[results.raw_15m == 4].est_input_tokens_mean.mean()
check("more raw candles cost more tokens", r8 > r4, f"{r4:.0f} -> {r8:.0f}")
if len(GRID["decision_history"]) > 1:
    h = results.groupby("decision_history").memory_tokens_mean.mean()
    check("a longer history window costs more memory tokens", h.iloc[-1] > h.iloc[0], h.round(0).to_dict())
check("all requests produced usage rows", (results.requests == len(CANDLES) * len(fx.symbols)).all())
if MODE == tk.MOCK:
    check("structured output valid in MOCK", (results.valid_rate == 1).all())

,est_input_tokens_mean,memory_tokens_mean
raw_15m,,
4,1905.0,274.0
8,1954.0,274.0
16,2054.0,274.0


,est_input_tokens_mean,memory_tokens_mean
candles_15m,,
16,1965.0,272.0
32,1977.0,277.0


,est_input_tokens_mean,memory_tokens_mean
decision_history,,
3,1957.0,260.0
5,1985.0,289.0


## Save for later comparison

Results go to `notebooks/agent_tests/results/` (git-ignored), stamped with the fixture version and mode, never into `fixtures/`.

In [5]:
results["fixture"] = fx.version; results["mode"] = MODE
print(tk.save_results(f"05_token_benchmark_{MODE.lower()}", results))
res = check.frame(); display(res); check.raise_if_failed()

/tmp/claude-0/-home-user-qqq-boxx-strategies/d20be8e8-2747-505a-9165-041a5d5e0f58/scratchpad/nbres/05_token_benchmark_mock_20261010T092916Z.csv


,check,result,detail
0,more raw candles cost more tokens,PASS,1905 -> 1954
1,a longer history window costs more memory tokens,PASS,"{3: 260.0, 5: 289.0}"
2,all requests produced usage rows,PASS,
3,structured output valid in MOCK,PASS,


**Reading the numbers**
- `usage_source=provider` with a NULL category means the provider did not report it.
- MOCK token counts equal the real prompt size, so the knob comparison transfers; the
  model's own output length and reasoning tokens only appear in REAL_LLM runs.
- Cost is an estimate from the configurable price table (`pricing.py`).